# NV Maser Digital Twin — Exploration

Interactive exploration of the digital twin simulation: field snapshots,
coil influence matrices, disturbance spectra, and training dynamics.

In [ ]:
%matplotlib inline
import sys
import os

# Ensure the repo src/ directory is on the path
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
src_path = os.path.join(repo_root, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import numpy as np
import matplotlib.pyplot as plt

from nv_maser.config import SimConfig, GridConfig
from nv_maser.physics.environment import FieldEnvironment
from nv_maser.viz.plots import (
    plot_field_snapshot,
    plot_coil_influence,
    plot_disturbance_spectrum,
    plot_training_history,
)

# Use a small 32x32 grid for fast interactive exploration
config = SimConfig()
config.grid = GridConfig(size=32)
print(f'Grid : {config.grid.size}x{config.grid.size}')
print(f'Coils: {config.coils.num_coils}')
print(f'B0   : {config.field.b0_tesla} T')
print(f'Max disturbance: {config.disturbance.max_amplitude_tesla} T')

In [ ]:
env = FieldEnvironment(config)
disturbance = env.disturbance_gen.generate()
distorted = env.base_field + disturbance

# Zero-current baseline: show the raw distorted field before any correction
zero_currents = np.zeros(config.coils.num_coils, dtype=np.float32)
correction = env.coils.compute_field(zero_currents)
net = distorted + correction

fig = plot_field_snapshot(
    distorted=distorted,
    correction=correction,
    net=net,
    grid=env.grid,
    coil_array=env.coils,
)
plt.show()

In [ ]:
fig = plot_coil_influence(env.coils)
plt.show()

In [ ]:
disturbance = env.disturbance_gen.generate()
fig = plot_disturbance_spectrum(disturbance)
plt.show()

## Training (quick demo)

Run a short training session (50 samples, 3 epochs) to verify the full
pipeline end-to-end and inspect the loss curves.

In [ ]:
from nv_maser.config import TrainingConfig
from nv_maser.model.training import Trainer

train_config = SimConfig()
train_config.grid = GridConfig(size=32)
train_config.training = TrainingConfig(
    num_samples=50,
    epochs=3,
    batch_size=10,
    val_split=0.2,
    checkpoint_dir='../checkpoints/notebook_demo',
)
train_config.device = 'cpu'

trainer = Trainer(train_config)
history = trainer.train()

fig = plot_training_history(history)
plt.show()
print(f'Best val loss: {trainer.best_val_loss:.4e}')